![Databricks Academy](./Includes/images/common/db-academy.png)

# 05 - Developing a Simple Pipeline

In this demonstration, we will build a simple Apache Spark™ Declarative Pipeline **from scratch** using the new **Lakeflow Pipeline Editor** with declarative SQL. Rather than starting from pre-written pipeline files, you will create a new, empty ETL pipeline, configure its settings, and then **copy the SQL code for each layer of the medallion architecture directly into your pipeline file**.


### Learning Objectives

By the end of this lesson, you will be able to:
- Create a new Apache Spark™ Declarative Pipeline in the Lakeflow Pipeline Editor.
- Describe the SQL syntax used to create bronze, silver, and gold pipeline objects.
- Add a configuration parameter to reference a raw data source volume.
- Create, execute, and monitor a Spark Declarative Pipeline.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this course.

This cell will also reset your `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7fc2d8ba-db02-4155-aee3-65cb7aad5a55
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7fc2d8ba-db02-4155-aee3-65cb7aad5a55
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.

ℹ️ Non-Vocareum workspace detected. Setting up catalog.
ℹ️ Using default catalog name: 'learn_build_data_pipelines_sdp_joemar_brillo'.
✅ Catalog 'learn_build_data_pipelines_sdp_joemar_brillo' already exists. Using this catalog.



  STEP 1: Verifying catalog exists: learn_build_data_pipelines_sdp_joemar_brillo
  Catalog 'learn_build_data_pipelines_sdp_joemar_brillo' exists.

  STEP 2: Setting up 3 schema(s) in catalog: learn_build_data_pipelines_sdp_joemar_brillo
  [1/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  STEP 1: Validating volume folder path...
  Found: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/joemar.brillo@outlook.com/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/data/customers

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4: Copying 1 file(s) to target volume...
  Source:      /Workspace/Users/joemar.brill

Information,Value
Your Catalog:,learn_build_data_pipelines_sdp_joemar_brillo
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


1. Run the cell below to view the value of the `source_volume_path` variable.

   Confirm that the value references your **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_1_bronze.source** volume. This volume contains the **orders**, **status**, and **customer** directories with the raw JSON files. You will use this path when configuring your pipeline.

   **EXAMPLE PATH**: `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source`

In [0]:
%python
print(source_volume_path)

/Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source


## B. Create the Spark Declarative Pipeline

In this section, you will create a Spark Declarative Pipeline using the Lakeflow Pipeline Editor, configure its settings, and add a configuration parameter that points to your raw source volume.

### B1. Enable the Lakeflow Pipelines Editor
##### If you completed `02 Demo - Course Setup and Creating a Pipeline`, you have already enabled the required setting and can skip this subsection.


1. Complete the following steps to enable the **Lakeflow Pipelines Editor**:
      - **NOTE:** This may already be enabled in your workspace.

   a. In the top-right corner, select your user icon ![User Lab Icon](./Includes/images/common/user_lab_circle_icon.png).

   b. Right-click on **Settings** and select **Open in New Tab**.

   c. Select **Developer**.

   d. Scroll to the bottom, enable **Lakeflow Pipelines Editor** if it is not already enabled, and select **Enable tabs for notebooks and files**.

   ![Lakeflow Pipeline Editor](./Includes/images/creating-a-pipeline/lakeflow-pipeline-editor.png)

   e. Refresh your browser to apply the changes.

### B2. Create a New ETL Pipeline

Complete the following steps to create your Spark Declarative Pipeline:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open Link in New Tab**.

2. In the new tab, select **Create → ETL Pipeline**.

   **NOTE:** If prompted to **Try the new Lakeflow Pipelines Editor**, choose **Enable Lakeflow Pipelines Editor**. This appears only if you did not complete the previous step.

3. At the top, complete the following:
   - Name your pipeline `05 - Developing a Simple Pipeline-add-your-name`
      
       **NOTE:** If prompted to rename root folder, Please rename it as well
   - Select your default **catalog** and **schema**:
        - **Catalog:** Your **learn_build_data_pipelines_sdp_YOUR_USERNAME** catalog
        - **Schema:** **sdp_1_bronze**

      **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. Rename the **transformations** folder to `orders`.

5. Rename the **my_transformation.sql** file to `orders_pipeline.sql`.

   **NOTE:** If your starter file is a Python file (`my_transformation.py`), delete it and create a new **SQL** file named `orders_pipeline.sql` instead. This demonstration focuses on SQL.

6. Leave the **Lakeflow Pipelines Editor** page open.

### B3. Explore and Modify Pipeline Settings

1. Select the **Settings** icon (gear icon) in your Pipeline Editor tab.
    - A panel will open on the right.

2. **Pipeline Settings** - Review the pipeline details:
    - Pipeline ID, type, name, mode, creator, owner, and run as.
    - Leave these as is.

3. **Code Assets** - This section automatically references all included files in your project (your new `orders` folder and `orders_pipeline.sql` file).

4. **Default Location for Data Assets**

    - Select **Edit the catalog and schema** and confirm the following:

        | Field | What to Select |
        |-------|---------------|
        | **Default catalog** | Your **learn_build_data_pipelines_sdp_YOUR_USERNAME** catalog |
        | **Default schema** | **sdp_1_bronze** |

    - Select **Save** if you updated these.

    > **NOTE:** With Apache Spark™ Declarative Pipelines you can publish streaming tables and materialized views to any catalog and schema. You are not restricted to the default catalog and schema.

5. **Compute** - Sets the pipeline compute

    a. Select the **Edit** icon and confirm that **Serverless** is selected.

    b. Deselect **Serverless** and review the available compute options, cluster policies, and tags.

    c. **Reselect Serverless** and navigate back to the pipeline settings.


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    REQUIRED - Add a Configuration Parameter to Point to your Source Volume
  </strong>
  <div style="color:#333;">

6. Complete the following steps to add a `source` configuration parameter pointing to your raw data volume. Your pipeline code will reference this parameter to locate the source files.

   a. Run the cell below and copy the path to your source data volume.

   b. In **Settings**, under **Configuration**, select **Add configuration**.

   c. **Key** = `source`

   d. **Value** = Paste the volume path of your `learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_1_bronze.source` volume.
    - Example: `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source`

   e. Select **Save**.

**Use parameters with pipelines**:
[AWS](https://docs.databricks.com/aws/en/ldp/parameters) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/parameters) |
[GCP](https://docs.databricks.com/gcp/en/ldp/parameters)
  </div>
</div>


In [0]:
%python
print(source_volume_path)

/Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source


7. **Usage** - Review the **Usage** section. This is where you can assign a budget policy for Serverless compute. Tags are applied to compute activity and logged in your billing records.

8. **Notifications** - Enables you to add notifications on pipeline runs.

9. **Advanced Settings**

     a. Expand **Advanced Settings** and select **Edit advanced settings**. Review the following options:

    - **Channel** - controls the pipeline runtime version. Leave as **Current**.

    - **Event Logs** - optionally write pipeline audit logs, data quality checks, and lineage to a table. Leave this deselected for now.
10. Select anywhere in the code editor to close the **Settings** panel.

You now have an empty pipeline configured with a default catalog, default schema, and a `source` configuration parameter. Next, you will add the pipeline code.

## C. Build the Pipeline Code

In this section, you will build a simple medallion architecture by adding SQL to your `orders_pipeline.sql` file, one layer at a time:

1. **Bronze** - incrementally ingest the raw JSON orders files into a streaming table.
2. **Silver** - clean and type the bronze data in a second streaming table.
3. **Gold** - aggregate the silver data into a daily-orders materialized view.

> **NOTE:** You can also build pipelines with Python. This demonstration focuses on SQL.

### C1. Create the Bronze Streaming Table

1. Start by creating the **bronze streaming table** that incrementally ingests the raw JSON orders files from your source volume.

   Copy the SQL code below and paste it into your `orders_pipeline.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--------------------------------------------------------------------------------------------------------------
-- A. Create the bronze streaming table from the raw JSON files in your source volume
--------------------------------------------------------------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE sdp_1_bronze.orders_bronze_demo05
AS
SELECT
  *,
  current_timestamp() AS processing_time,   -- Records when each row was ingested
  _metadata.file_name AS source_file        -- Captures the source file name for lineage
FROM STREAM read_files(                      -- Performs incremental ingestion with checkpoints using Auto Loader
    "${source}/orders",                      -- Uses the 'source' configuration parameter set in the pipeline settings
    format => 'JSON'
);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code**
- `CREATE OR REFRESH STREAMING TABLE` creates the **sdp_1_bronze.orders_bronze_demo05** streaming table, the landing zone for the raw orders data.
- `FROM STREAM read_files("${source}/orders", ...)` uses **Auto Loader** for incremental ingestion and references the `source` configuration parameter you set in the pipeline settings.
- The `processing_time` and `source_file` columns are added for auditing and lineage.
- Because this is a streaming table, the pipeline **incrementally processes only new files** on each run.

### C2. Create the Silver Streaming Table

1. Next, create the **silver streaming table** that reads incrementally from the bronze table, then selects and cleans the relevant columns.

   Copy the SQL code below and paste it into your `orders_pipeline.sql` file, below the bronze code.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--------------------------------------------------------------------------------------------------------------
-- B. Create the silver streaming table from the bronze streaming table
--------------------------------------------------------------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE sdp_2_silver.orders_silver_demo05
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp,   -- Casts the order_timestamp to a TIMESTAMP type
  customer_id,
  notifications
FROM STREAM sdp_1_bronze.orders_bronze_demo05;      -- Incrementally reads from the bronze streaming table
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code**
- `CREATE OR REFRESH STREAMING TABLE` creates the **sdp_2_silver.orders_silver_demo05** streaming table.
- `FROM STREAM sdp_1_bronze.orders_bronze_demo05` reads **incrementally** from the bronze streaming table, so only new bronze rows flow through on each run.
- The query selects only the relevant columns and casts `order_timestamp` to a proper `TIMESTAMP` type.

### C3. Create the Gold Materialized View

1. Finally, create the **gold materialized view** that aggregates the silver data into daily order counts.

   Copy the SQL code below and paste it into your `orders_pipeline.sql` file, below the silver code.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--------------------------------------------------------------------------------------------------------------
-- C. Create the gold materialized view aggregating the silver table by order date
--------------------------------------------------------------------------------------------------------------
CREATE OR REFRESH MATERIALIZED VIEW sdp_3_gold.gold_orders_by_date_demo05
AS
SELECT
  date(order_timestamp) AS order_date,
  count(*) AS total_daily_orders
FROM sdp_2_silver.orders_silver_demo05   -- Aggregates the full silver table with optimizations where applicable
GROUP BY date(order_timestamp);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code**
- `CREATE OR REFRESH MATERIALIZED VIEW` creates the **sdp_3_gold.gold_orders_by_date_demo05** materialized view.
- It aggregates the silver table into **daily order counts** (`GROUP BY date(order_timestamp)`).
- A `MATERIALIZED VIEW` is used here instead of a streaming table because `GROUP BY` aggregations can't be incrementally computed as a streaming table. Databricks still optimizes recomputation where possible.
- **Incremental refresh for materialized views**:
[AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
[GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)

**Key Differences ST/MV**:
  - Bronze and Silver use `STREAMING TABLE` for incremental processing.
  - Gold uses `MATERIALIZED VIEW` because `GROUP BY` aggregations can't be incrementally computed as a streaming table.

## D. Run the Spark Declarative Pipeline

### D1. Dry Run the Pipeline
Using a dry run, you can **check for problems in a pipeline's source code** without waiting for tables to be created or updated.

This feature is useful when developing or testing pipelines because it lets you quickly find and fix errors in your pipeline, such as incorrect table or column names.


1. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Select **Dry Run** from the top navigation bar (if your screen is small, you might have to select the drop-down next to the **Run pipeline** button in the top navigation bar).</span>

    - Notice that the pipeline switches to **DRY RUN** mode in the left window and begins processing each step (this takes about ~1 minute).

    - After the dry run completes, you should see three items in the **Pipeline graph** (Select the graph icon in the right navigation bar if necessary):

      ![Graph](./Includes/images/developing-a-simple-pipeline/dry-run-pipeline-graph.png)

2. Explore the bottom window and observe the following columns:

    - **Catalog** - the catalog each object will write to

    - **Schema** - the schema each object will write to

    - **Type** - the object type

      ![Window](./Includes/images/developing-a-simple-pipeline/pipeline-dry-run-window.png)

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting a Pipeline Error
  </strong>
  <div style="color:#333;">

If your pipeline returns an error, verify the following in your <strong>Pipeline Settings</strong>:

  - Default catalog is set to your <strong>learn_build_data_pipelines_sdp_YOUR_USERNAME</strong> catalog

  - Default schema is set to <strong>sdp_1_bronze</strong>

  - The `source` configuration variable references your volume path: `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source`

  - All three SQL statements were pasted into your `orders_pipeline.sql` file

  </div>
</div>

### D2. Run the Spark Declarative Pipeline

1. Select the drop-down next to the **Run pipeline** button in the top navigation bar.

You will see two options:

| Update Type | Materialized View | Streaming Table |
|-------------|------------------|-----------------|
| **Run pipeline** - Refresh | Updates results to reflect the current results for the defining query. Will examine the costs, and perform an incremental refresh if it is more cost-efficient. | Processes new records through logic defined in streaming tables and flows. |
| **Run pipeline with full table refresh** - Full refresh of the pipeline | Updates results to reflect the current results for the defining query. | Clears data from streaming tables, clears state information (checkpoints) from flows, and reprocesses all records from the data source. |

2. Select **Run pipeline** and monitor each stage in the right window.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- A full refresh deletes all objects and checkpoints. Be careful when using.
- To prevent a full refresh from being triggered, set the table property: `pipelines.reset.allowed = false`
- **Pipeline refresh semantics**:
[AWS](https://docs.databricks.com/aws/en/ldp/updates#pipeline-refresh-semantics) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/updates#pipeline-refresh-semantics) |
[GCP](https://docs.databricks.com/gcp/en/ldp/updates#pipeline-refresh-semantics)

  </div>
</div>


### D3. Monitor the Pipeline Run

1. While the pipeline is running (~1 minute), observe the **Pipeline graph** on the right.

    As each object is created, a data flow appears and row counts update as processing completes.

    Expected results:
    - **174 rows** ingested from Raw JSON → Bronze → Silver
    - **7 rows** in the materialized view aggregation

### D4. Explore the Completed Pipeline

After the pipeline completes, explore the results:

1. Review the **Pipeline graph**. The graph visualizes the full data flow.

2. Review the bottom window for details on tables and materialized views created, including locations, durations, and row counts.

   a. Select the **Performance** tab to view performance metrics, then return to **Tables**.

   b. Select the **orders_bronze_demo05** streaming table to view its **data**, **Table metrics**, and **Performance**.

   c. Select the arrow to the left of **All tables** in the bottom window to return to the full pipeline objects list.

   d. Select the **gold_orders_by_date_demo05** materialized view and confirm it summarizes data by date as expected.
      - Use the navigation bar in the bottom window to explore the **Data**, **Columns**, **Table metrics**, and **Performance** tabs for the materialized view.

### D5. Explore the DAG (Pipeline Graph)

1. You can also select objects directly within the DAG (Pipeline Graph) to view the details in the bottom window.

### D6. Run the Pipeline Again

1. Select **Run pipeline** again and explore the updated results while it runs.

2. After the second run completes, note that no new rows were added to the streaming tables.

**This is expected since no new files were added to the data source and the checkpoints know the current data was already ingested**.

![Run 2](./Includes/images/developing-a-simple-pipeline/run-2-no-changes.png)

## E. Add a New File to Cloud Storage

1. Run the cell below to add a new JSON file (**01.json**) to your volume at:  `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source/orders`.

      This will simulate files being added to cloud storage.

In [0]:
%python
## Use the course seed data bundled in Includes/data
course_data_path = get_course_data_path()

## Land another JSON file to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{course_data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=2
)


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/joemar.brillo@outlook.com/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/data/orders

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/orders

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4: Copying 2 file(s) to target volume...
  Source:      /Workspace/Users/joemar.brillo@outlook.com/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/data/orders
  Destination: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/orders
  [1/2] Checking: 00.json... EXISTS at destination. Skipping.
  [2/2] Checking: 01.json... NOT found at destination. Copied successfully.

  COMPLETE: Copied 1 new file(s), skipped 1.



2. Complete the following steps to view the new file in your volume:

   a. Select the **Catalog** icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) from the left navigation pane.

   b. Expand your **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_1_bronze.source** volume.

   c. Expand the **orders** directory.

   d. You should see two files in your volume: **00.json** and **01.json** (refresh if necessary).

3. Run the cell below to view the data in the new **/orders/01.json** file. Notice the following:

   - The **01.json** file contains new orders.
   - The **01.json** file has 25 rows.

In [0]:
%python
spark.sql(f'''
  SELECT *
  FROM json.`{source_volume_path}/orders/01.json`
''').display()

customer_id,notifications,order_id,order_timestamp
23180,Y,75297,1640996822
23082,Y,75298,1641000470
23550,Y,75299,1641000707
23362,Y,75300,1641002550
23210,N,75301,1641003380
23489,N,75302,1641004093
23328,Y,75303,1641004638
23954,Y,75304,1641009857
23648,Y,75305,1641011274
23310,Y,75306,1641015380


4. Go back to the **orders_pipeline.sql** file and select **Run Pipeline** to execute your ETL pipeline again with the new file.

   - Watch the pipeline run and notice only **25 rows** are added to the bronze and silver tables.

   - This happens because:
      - The pipeline has already processed the initial **00.json** file (174 rows)
      - It now reads only the new **01.json** file (25 rows)
      - New rows are appended to the **streaming tables**
      - The materialized view is **incrementally recomputed** using the latest data (should have a total of **8 rows**)

#### Checkpoint
![Final](./Includes/images/developing-a-simple-pipeline/demo2-final.png)

## F. Exploring Your Streaming Tables

### F1. View the Tables

1. View the new streaming tables and materialized view in your catalog. Complete the following:

   a. Select the catalog icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) in the left navigation pane.

   b. Expand your **learn_build_data_pipelines_sdp_YOUR_USERNAME** catalog.

   c. Expand the schemas **sdp_1_bronze**, **sdp_2_silver**, and **sdp_3_gold**.
      - Notice that the two streaming tables and materialized view are correctly placed in your schemas.

         - Streaming Bronze Table: **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_1_bronze.orders_bronze_demo05**

         - Streaming Silver Table: **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_2_silver.orders_silver_demo05**

         - Gold Materialized View: **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_3_gold.gold_orders_by_date_demo05**

2. Run the cell below to view the data in the **learn_build_data_pipelines_sdp_YOUR_USERNAME.sdp_1_bronze.orders_bronze_demo05** table.

   Before you run the cell, how many rows should this streaming table have?

   Notice the following:
      - The table contains 199 rows (**00.json** had 174 rows, and **01.json** had 25 rows).
      - In the **source_file** column you can see the exact file the rows were ingested from.
      - In the **processing_time** column you can see the exact time the rows were ingested.

In [0]:
select current_schema(), current_catalog();

current_schema(),current_catalog()
sdp_1_bronze,learn_build_data_pipelines_sdp_joemar_brillo


In [0]:
select current_catalog(), current_schema();
show tables like '%bronze%'


current_catalog(),current_schema()
learn_build_data_pipelines_sdp_joemar_brillo,sdp_1_bronze


In [0]:
SELECT *
FROM learn_build_data_pipelines_sdp_joemar_brillo.sdp_1_bronze.orders_bronze;

customer_id,notifications,order_id,order_timestamp,_rescued_data,processing_time,source_file
23094,Y,75123,1640392092,null,2026-11-07T01:38:46.993Z,00.json
23457,N,75124,1640392500,null,2026-11-07T01:38:46.993Z,00.json
23564,Y,75125,1640394862,null,2026-11-07T01:38:46.993Z,00.json
23392,N,75126,1640396067,null,2026-11-07T01:38:46.993Z,00.json
23101,Y,75127,1640399066,null,2026-11-07T01:38:46.993Z,00.json
23466,N,75128,1640404853,null,2026-11-07T01:38:46.993Z,00.json
23834,Y,75129,1640407272,null,2026-11-07T01:38:46.993Z,00.json
23852,Y,75130,1640419989,null,2026-11-07T01:38:46.993Z,00.json
23483,Y,75131,1640422131,null,2026-11-07T01:38:46.993Z,00.json
23821,N,75132,1640423697,null,2026-11-07T01:38:46.993Z,00.json


### F2. View the Table History

1. Run the code below to view the history of the **orders_bronze_demo05** streaming table.

      Notice the following:

      - In the **operation** column, the last two updates are **STREAMING UPDATE**, not `WRITE` or `MERGE`. This confirms the table is being written to incrementally by a streaming query.

      - Expand the **operationParameters** values for the last two updates. Notice both use `"outputMode": "Append"` and each has an incrementing **epochId** (`0` for the initial load, `1` for the incremental update).

      - Find the **operationMetrics** column. Expand the values for the last two updates. Observe the following:

         - It displays various metrics for the streaming update: **numRemovedFiles, numOutputRows, numOutputBytes, and numAddedFiles**.

         - In the `numOutputRows` values
            - **174 rows** were added in the first update
            -  **25 rows** in the second.
            - If this were a batch pipeline doing a full re-read, the second run would show 199 rows. The fact that it shows only 25 proves that only new data was processed.

In [0]:
DESCRIBE HISTORY sdp_1_bronze.orders_bronze;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-07T01:38:49.000Z,74546832009708,joemar.brillo@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""e1c4e436-fd3f-4f17-a832-e998a3c78540"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ba296e62-cfb6-4d68-99f0-61f38c8c3385"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-9715d3f8-4099-40c6-8212-45d1ce6c4efd""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(178197117461980),51fee876-92fa-455a-bc7c-e6dd713cc241,1107-005941-qyzlqn3c-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 174, numOutputBytes -> 2997)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


## G. Viewing Spark Declarative Pipelines with the Pipelines UI

After building and running your pipeline using the **orders_pipeline.sql** file in the steps above, you can view the pipeline(s) you created in your workspace via the **Jobs and Pipelines** UI.

1. Complete the following steps to view the pipeline you created:

   a. In the main applications navigation pane on the far left (you may need to expand it by selecting the ![Expand Navigation Pane](./Includes/images/developing-a-simple-pipeline/expand_main_navigation.png) icon at the top left of your workspace) right-click on **Jobs & Pipelines** and select **Open Link in a New Tab**.

   b. This should take you to the pipelines you have created. You should see your **05 - Developing a Simple Pipeline-your name** pipeline.

   c. Select your **05 - Developing a Simple Pipeline-your name**.

   d. Under your pipeline name, select the drop-down with the timestamp. Here you can view the **Pipeline graph** and other metrics for each run of the pipeline.

   e. Close the pipeline UI tab you opened.

   ![Jobs & Pipelines](./Includes/images/developing-a-simple-pipeline/demo_2_view_in_jobs_pipelines.png)

## Additional Resources

- **Apache Spark™ Declarative Pipelines documentation**:
[AWS](https://docs.databricks.com/aws/en/dlt/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/) |
[GCP](https://docs.databricks.com/gcp/en/dlt/)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
